1. Prepare Your Dataset

Your dataset appears to be in a format similar to the CoNLL-2003 format, where each token is followed by its corresponding label. Ensure your dataset is properly formatted and split into training, validation, and test sets

Example format:

-DOCSTART- -X- O

Contact -X- _ O

www.linkedin.com -X- _ O
...

2. Install Required Libraries

Make sure you have the necessary libraries installed. You will need transformers, datasets, and torch.

In [4]:
#%pip install transformers datasets torch

3. Load the Pre-trained Model and Tokenizer

Load the xlm-roberta-large-finetuned-conll03-english model and tokenizer using the transformers library.

In [5]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "xlm-roberta-large-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

Some weights of the model checkpoint at xlm-roberta-large-finetuned-conll03-english were not used when initializing XLMRobertaForTokenClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


4. Prepare the Dataset for Training

Convert your dataset into a format that can be used by the transformers library. You can use the datasets library to load and preprocess your data.

In [6]:
from datasets import Dataset, Features, Sequence, ClassLabel, Value

# Function to load dataset
def load_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    tokens = []
    labels = []
    current_tokens = []
    current_labels = []

    for line in lines:
        if line.strip() == "":
            if current_tokens:
                tokens.append(current_tokens)
                labels.append(current_labels)
                current_tokens = []
                current_labels = []
        else:
            parts = line.strip().split()
            current_tokens.append(parts[0])
            current_labels.append(parts[-1])

    return {"tokens": tokens, "labels": labels}

# Load your dataset
train_data = load_dataset("C:/Users/Acer/Desktop/ARSwithPredictiveAnalytics/Data-Training/conll-2003-traindata.conll")
val_data = load_dataset("C:/Users/Acer/Desktop/ARSwithPredictiveAnalytics/Data-Training/conll-2003-evaldata.conll")

# Print the structure of train_data to verify
print("Train data structure:", train_data)

# Extract unique labels
def get_label_names(dataset):
    label_names = set()
    for labels in dataset["labels"]:  # Iterate over the "labels" key
        label_names.update(labels)    # Add all labels to the set
    return sorted(list(label_names))  # Convert to a sorted list

label_names = get_label_names(train_data)
print("Label names:", label_names)
print("Number of labels:", len(label_names))
# Define features
features = Features({
    "tokens": Sequence(Value("string")),
    "labels": Sequence(ClassLabel(names=label_names))
})

# Convert to datasets.Dataset
train_dataset = Dataset.from_dict(train_data, features=features)
val_dataset = Dataset.from_dict(val_data, features=features)

Train data structure: {'tokens': [['Contact', 'www.linkedin.com', '/', 'in', '/', 'hazel-whitfield', '(', 'LinkedIn', ')', 'Top', 'Skills', 'Responsive', 'Web', 'Design', 'Skilled', 'Multi-tasker', 'Web', 'Development', 'Certifications', 'Lean', 'Six', 'Sigma', 'Yellow', 'Belt', 'Certification', 'Full', 'Stack', 'Web', '+', 'Mobile', 'Development', 'Lean', 'Six', 'Sigma', 'DMAIC', 'Certification', 'Lean', 'Six', 'Sigma', 'Project', 'Management', 'Certification', 'Honors-Awards', 'Honor', 'Certificate', '-', 'Front', 'End', 'Web', 'and', 'Mobile', 'Development', 'Hazel', 'Whitfield', 'Accounting', 'Professional', 'Greater', 'Orlando', 'Experience', 'CareerSource', 'Central', 'Florida', 'Senior', 'Accounting', 'Specialist', 'September', '2024', '-', 'Present', '(', '6', 'months', ')', 'R.A.', 'Simasek', ',', 'P.A.', 'Office', 'Manager', '/', 'Bookkeeper', 'May', '2024', '-', 'July', '2024', '(', '3', 'months', ')', 'CareerSource', 'Central', 'Florida', '3', 'years', '4', 'months', 'Senio

5. Tokenize the Dataset

Tokenize the dataset using the tokenizer. Ensure that the labels are aligned with the tokenized inputs.

In [7]:
def tokenize_and_align_labels(examples):
    # Tokenize the inputs
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,          # Enable truncation
        padding=True,             # Enable padding
        is_split_into_words=True, # Indicate that inputs are pre-tokenized
        max_length=512,           # Set a maximum length (adjust as needed)
        return_tensors="pt",      # Return PyTorch tensors
    )

    # Align labels with tokenized inputs
    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective words
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens (e.g., [CLS], [SEP], padding) get a label of -100
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # New word, assign the corresponding label
                label_ids.append(label[word_idx])
            else:
                # Same word as previous token (subword tokenization), assign -100
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    # Add labels to the tokenized inputs
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

train_dataset = train_dataset.map(tokenize_and_align_labels, batched=True)
val_dataset = val_dataset.map(tokenize_and_align_labels, batched=True)

# Check the first example in the tokenized dataset
print("Tokenized input IDs:", train_dataset[0]["input_ids"])
print("Tokenized labels:", train_dataset[0]["labels"])

Map: 100%|██████████| 18/18 [00:00<00:00, 277.67 examples/s]

Tokenized input IDs: [0, 27627, 1426, 5, 10187, 297, 73, 5, 277, 248, 23, 248, 256, 5076, 9, 434, 11400, 28394, 15, 92502, 1388, 4792, 132829, 7, 6, 94399, 5844, 4002, 11724, 132829, 297, 19335, 9, 1073, 1728, 4002, 55441, 83991, 5256, 124529, 84247, 173328, 146762, 12628, 18, 83991, 1363, 9312, 6512, 2594, 4002, 997, 19745, 55441, 124529, 84247, 173328, 391, 131802, 441, 83991, 1363, 124529, 84247, 173328, 27331, 25924, 83991, 1363, 58744, 7, 9, 284, 19364, 7, 58744, 83991, 67, 20, 43643, 18878, 4002, 136, 19745, 55441, 1391, 5076, 127325, 28394, 61417, 214, 61928, 32774, 56, 121839, 135634, 20731, 56, 211235, 15881, 52888, 49952, 61417, 214, 171322, 6088, 387, 2357, 20, 97721, 15, 305, 21775, 1388, 627, 5, 284, 5, 602, 1510, 343, 6, 4, 436, 5, 284, 5, 12133, 30195, 248, 27432, 184308, 4347, 387, 2357, 20, 20414, 387, 2357, 15, 138, 21775, 1388, 20731, 56, 211235, 15881, 52888, 138, 5369, 201, 21775, 49952, 61417, 214, 171322, 20414, 64371, 20, 7582, 72392, 15, 106, 6602, 190, 21775, 

6. Set Up Training Arguments

Define the training arguments using TrainingArguments.

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,  # Adjust based on your GPU memory
    per_device_eval_batch_size=8,   # Adjust based on your GPU memory
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_steps=10,
    save_total_limit=2,
    fp16=True,  # Enable mixed precision training if using a compatible GPU
)

c:\Users\Acer\AppData\Local\Programs\Python\Python39\lib\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


7. Define the Trainer

Set up the Trainer class to handle the training loop.

In [9]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

C:\Users\Acer\AppData\Local\Temp\ipykernel_7920\1666238700.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
from transformers import AutoModelForTokenClassification
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Step 1: Check the number of unique labels
print("Label names:", label_names)
print("Number of labels:", len(label_names))

# Step 2: Update the model's num_labels
model = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-large-finetuned-conll03-english",
    num_labels=len(label_names),  # Set num_labels to the number of unique labels
)

# Step 3: Verify label indices
all_labels = [label for sublist in train_data["labels"] for label in sublist]
max_label_index = np.max(all_labels)
print("Maximum label index:", max_label_index)

# Step 4: Reindex labels if necessary
if max_label_index >= len(label_names):
    print("Reindexing labels...")
    label_encoder = LabelEncoder()
    encoded_labels = label_encoder.fit_transform(all_labels)
    encoded_labels = np.array(encoded_labels).reshape(len(train_data["labels"]), -1)
    train_data["labels"] = encoded_labels.tolist()

# Step 5: Re-run training
trainer.train()

Label names: ['B-B-ADDRESS', 'B-B-CERTIFICATE', 'B-B-EDUCATION', 'B-B-SKILLS', 'B-I-ADDRESS', 'B-I-CERTIFICATE', 'B-I-EDUCATION', 'B-I-SKILLS', 'B-O', 'EXPERIENCE', 'I-B-ADDRESS', 'I-B-CERTIFICATE', 'I-B-EDUCATION', 'I-B-SKILLS', 'I-I-ADDRESS', 'I-I-CERTIFICATE', 'I-I-EDUCATION', 'I-I-SKILLS', 'I-O', 'O']
Number of labels: 20


RuntimeError: Error(s) in loading state_dict for XLMRobertaForTokenClassification:
	size mismatch for classifier.weight: copying a param with shape torch.Size([8, 1024]) from checkpoint, the shape in current model is torch.Size([20, 1024]).
	size mismatch for classifier.bias: copying a param with shape torch.Size([8]) from checkpoint, the shape in current model is torch.Size([20]).
	You may consider adding `ignore_mismatched_sizes=True` in the model `from_pretrained` method.

8. Train the Model

In [ ]:
trainer.train()

9. Evaluate the Model

In [ ]:
trainer.evaluate()

10. Save the Model

Save the fine-tuned model for future use.

In [ ]:
model.save_pretrained("./fine-tuned-model")
tokenizer.save_pretrained("./fine-tuned-model")

11. Inference

You can now use the fine-tuned model for inference on new data.

In [ ]:
from transformers import pipeline

ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer)
results = ner_pipeline("Your input text here")
print(results)